In [1]:

# ==================================================
# 5CLOUD - VELIB DATA PLATFORM
# Notebook Azure Synapse : Bronze -> Silver -> Gold
# ==================================================

BATCH_ID = "0610bc1b-5d65-462b-8a84-335af2737c68"

STORAGE_ACCOUNT = "stvelibchifa2026"
CONTAINER = "velib-data"

ADLS_ROOT = (
    f"abfss://{CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
)

BRONZE_PATH = (
    f"{ADLS_ROOT}/bronze/velib/batch_id={BATCH_ID}"
)

SILVER_PATH = (
    f"{ADLS_ROOT}/silver/velib/batch_id={BATCH_ID}"
)

GOLD_GLOBAL_PATH = (
    f"{ADLS_ROOT}/gold/velib/global/batch_id={BATCH_ID}"
)

GOLD_COMMUNES_PATH = (
    f"{ADLS_ROOT}/gold/velib/communes/batch_id={BATCH_ID}"
)

print("=== 5CLOUD - AZURE SYNAPSE ===")
print("Batch :", BATCH_ID)
print("Bronze :", BRONZE_PATH)
print("Silver :", SILVER_PATH)
print("Gold Global :", GOLD_GLOBAL_PATH)
print("Gold Communes :", GOLD_COMMUNES_PATH)
print("Configuration Synapse prête !")


=== 5CLOUD - AZURE SYNAPSE ===
Batch : 0610bc1b-5d65-462b-8a84-335af2737c68
Bronze : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/bronze/velib/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Silver : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/silver/velib/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Gold Global : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/gold/velib/global/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Gold Communes : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/gold/velib/communes/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Configuration Synapse prête !


In [ ]:
print("=== SYNAPSE : CONFIGURATION MANIFESTE BRONZE ===")

# Chemin du manifeste associé au lot
MANIFEST_PATH = f"{BRONZE_PATH}/manifest.json"

print("Manifeste Bronze :", MANIFEST_PATH)

# Lecture à effectuer dans le workspace Synapse
def load_collection_time(spark, manifest_path):
    import json
    from datetime import datetime

    # Lecture du manifeste depuis ADLS
    manifest_rows = spark.read.text(
        manifest_path,
        wholetext=True
    ).collect()

    manifest = json.loads(
        manifest_rows[0]["value"]
    )

    collection_time = datetime.fromisoformat(
        manifest["manifest_written_at_utc"].replace(
            "Z", "+00:00"
        )
    )

    return collection_time


print("Fonction de lecture du manifeste préparée !")


In [ ]:
from pyspark.sql import functions as F

print("=== SYNAPSE : LECTURE BRONZE ===")

# Chemin des fichiers JSON de Chifa
BRONZE_PAGES_PATH = f"{BRONZE_PATH}/pages/page_*.json"

# Lecture des 16 pages JSON dans Azure
pages_df = (
    spark.read
    .option("multiLine", "true")
    .json(BRONZE_PAGES_PATH)
)

# Les stations se trouvent dans la clé "results"
stations_df = (
    pages_df
    .select(F.explode("results").alias("station"))
    .select("station.*")
)

print("Stations Bronze :", stations_df.count())
print("Nombre de colonnes :", len(stations_df.columns))

stations_df.select(
    "stationcode",
    "name",
    "capacity",
    "numbikesavailable",
    "numdocksavailable"
).show(5, truncate=False)

assert stations_df.count() > 0

print("Lecture Bronze Azure réussie !")


In [ ]:
import json
from datetime import datetime

from src.platform_project.cleaning import clean_velib_data
from src.platform_project.quality import check_data_quality
from src.platform_project.silver_history import prepare_silver_history

print("=== SYNAPSE : TRANSFORMATION SILVER ===")

# Valeur de référence du manifeste Bronze
# À remplacer ensuite par la lecture du manifeste ADLS
#collection_time = datetime.fromisoformat(
#    "2026-10-28T21:35:06.283930+00:00"
#)

collection_time = load_collection_time(
    spark,
    MANIFEST_PATH
)

print("Date du manifeste :", collection_time)

# Nettoyage Silver
silver_df = clean_velib_data(
    stations_df,
    BATCH_ID,
    collection_time
)

# Contrôle qualité
valid_df, rejected_df = check_data_quality(silver_df)

# Déduplication
silver_history_df = prepare_silver_history(valid_df)

print("Stations Silver :", silver_df.count())
print("Stations valides :", valid_df.count())
print("Stations rejetées :", rejected_df.count())
print("Stations après déduplication :", silver_history_df.count())

print("Transformation Silver Synapse terminée !")


In [ ]:

from src.platform_project.aggregations import (
    build_gold_global,
    build_gold_communes
)

print("=== SYNAPSE : AGRÉGATIONS GOLD ===")

# Calcul des KPI globaux
gold_global_df = build_gold_global(silver_history_df)

# Calcul des KPI par commune
gold_communes_df = build_gold_communes(silver_history_df)

print("Nombre de lignes Gold Global :", gold_global_df.count())
print("Nombre de communes :", gold_communes_df.count())

print("\n--- KPI GLOBAUX ---")
gold_global_df.show(truncate=False)

print("\n--- TOP 10 COMMUNES ---")
gold_communes_df.orderBy(
    F.desc("total_bikes")
).show(10, truncate=False)

print("Agrégations Gold Synapse terminées !")


In [2]:
print("=== SYNAPSE : ÉCRITURE SILVER & GOLD ===")

# Sécurité : ne pas lancer les écritures
# tant que l'infrastructure n'est pas prête.
ENABLE_AZURE_WRITE = False

if ENABLE_AZURE_WRITE:

    from src.platform_project.azure_storage import (
        write_parquet_to_adls,
        read_parquet_from_adls
    )

    # Vérifier les résultats avant publication
    assert silver_history_df.count() > 0
    assert gold_global_df.count() == 1
    assert gold_communes_df.count() > 0

    # Enregistrer Silver
    write_parquet_to_adls(
        silver_history_df,
        SILVER_PATH
    )

    # Enregistrer Gold Global
    write_parquet_to_adls(
        gold_global_df,
        GOLD_GLOBAL_PATH
    )

    # Enregistrer Gold Communes
    write_parquet_to_adls(
        gold_communes_df,
        GOLD_COMMUNES_PATH
    )

    # Vérifier les fichiers publiés
    saved_silver = read_parquet_from_adls(
        spark, SILVER_PATH
    )

    saved_global = read_parquet_from_adls(
        spark, GOLD_GLOBAL_PATH
    )

    saved_communes = read_parquet_from_adls(
        spark, GOLD_COMMUNES_PATH
    )

    assert saved_silver.count() == silver_history_df.count()
    assert saved_global.count() == 1
    assert saved_communes.count() == gold_communes_df.count()

    print("Écriture et relecture Azure réussies !")

else:
    print("Écriture Azure désactivée.")
    print("En attente du workspace et des accès Synapse.")


=== SYNAPSE : ÉCRITURE SILVER & GOLD ===
Écriture Azure désactivée.
En attente du workspace et des accès Synapse.
